In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from dotenv import load_dotenv
import os

import warnings

from Crime_Data_Preprocessing_Pipeline import preprocess, table_structure, data_split, data_cleaning, feature_engineering
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

# BRONZE LAYER
_________________________________________________________________________________________________________
### LOAD (TO BRONZE LAYER)
Connect to the API and fetch the data and save it to the bronze layer


In [10]:
## Connect to the API and fetch the data
# 1. Load variables from the .env file into the system environment
load_dotenv()

# 2. Access variables using os.getenv()
my_secret = os.getenv("EASYDATA_API_KEY")
EASYDATA_API_URL = "https://www.easydata.co.za/api/v3/"

In [12]:
from quantec.easydata.client import Client
# All parameters and defaults
client = Client(
    api_key= my_secret,  # default: uses EASYDATA_API_KEY env var
    api_url=EASYDATA_API_URL,  # default: EASYDATA_API_URL or "https://www.easydata.co.za/api/v3"
    use_cache=True,
    cache_dir="cache",
)

In [62]:
# Extract my data using the selection_pk and frequency parameters (M = Monthly)
df1 = client.get_data(selection_pk = 19053, freq = "M")
df2 = client.get_data(selection_pk = 19058, freq = "M")
df3 = client.get_data(selection_pk = 19059, freq = "M")

In [13]:
# Load the data to the Bronze Layer
df1.to_excel(r"C:/Users/amare/OneDrive/Desktop/Crime Hotspots Capstone Project/datasets/Data Lakehouse/1. Bronze Layer/Crime Data/1st Batch.xlsx", index=False)
df2.to_excel(r"C:/Users/amare/OneDrive/Desktop/Crime Hotspots Capstone Project/datasets/Data Lakehouse/1. Bronze Layer/Crime Data/2nd Batch.xlsx", index=False)
df3.to_excel(r"C:/Users/amare/OneDrive/Desktop/Crime Hotspots Capstone Project/datasets/Data Lakehouse/1. Bronze Layer/Crime Data/3rd Batch.xlsx", index=False)


### EXTRACT (FROM BRONZE LAYER)
Extract the data from the bronze layer and load it into a dataframe for transformation

In [2]:
df1 = pd.read_excel(r"C:/Users/amare/OneDrive/Desktop/Crime Hotspots Capstone Project/datasets/Data Lakehouse/1. Bronze Layer/Crime Data/1st Batch.xlsx")
df2 = pd.read_excel(r"C:/Users/amare/OneDrive/Desktop/Crime Hotspots Capstone Project/datasets/Data Lakehouse/1. Bronze Layer/Crime Data/2nd Batch.xlsx")
df3 = pd.read_excel(r"C:/Users/amare/OneDrive/Desktop/Crime Hotspots Capstone Project/datasets/Data Lakehouse/1. Bronze Layer/Crime Data/3rd Batch.xlsx")

# SILVER LAYER
## TRANSFORM
Transform the data by applying the necessary transformations such as data cleaning, feature engineering, and data splitting. This can be done using pipelines to ensure that the transformations are applied consistently and efficiently.

In [3]:
# The pipelines take in a list of dataframes as input and apply the transformations to each dataframe in the list. The output is a single dataframe that has been transformed and is ready for loading into the silver layer.
X = [df1, df2, df3]

In [4]:
preprocess_pipe = Pipeline([
    ("Preprocessor", preprocess())
    ])
table_str_pipe = Pipeline([
    ("Table Structure", table_structure())
    ])

In [5]:
data = preprocess_pipe.fit_transform(X)
data = table_str_pipe.fit_transform(X)

In [6]:
data_split_pipe = Pipeline([
    ("Split Data", data_split()),
    ("Clean Data", data_cleaning()),
    ("New Cols", feature_engineering())
    ])
crime_data = data_split_pipe.fit_transform(data)

### LOAD TO SILVER LAYER

In [8]:
crime_data.to_excel(r"C:/Users/amare/OneDrive/Desktop/Crime Hotspots Capstone Project/datasets/Data Lakehouse/2. Silver Layer/Crime Data/cleaned_crime_data.xlsx", index=False)

### PLAN
Read other datasets and consolidate them
1. Explore Data Distributions and do feature selection
2. Train and Test Split
3. Encode the features
4. Normalize the values
5. Select and Train Model
6. Evaluate Model
7. Tune Parameters
8. Evaluate Model
9. Insights